# 🤖 Notebook 02: Modelagem Híbrida, Treinamento com GPU (CUDA) e Avaliação Top-K
**Projeto Aplicado III — Ciência de Dados e Inteligência Artificial (Universidade Presbiteriana Mackenzie)**  
**Tema:** Sistema de Recomendação de Espécies Nativas para Reflorestamento e Arborização Urbana (ODS 15)  
**Grupo:** Fernanda Pauli de Oliveira (10736757), Henrique Sarmento (10738262), Milena Dias Gouveia (10746541), Nicolly Falcão da Silva (10746134)  

---

## 📌 Objetivos deste Notebook
1. **Ambiente Acelerado por Hardware (GPU / CUDA):** Utilizar aceleração tensorial em GPU (**NVIDIA GeForce RTX 3060**) para treinamento de Fatoração Matricial com Feedback Implícito (CF-ALS).
2. **Divisão Espacial Estrita (*Spatial Cluster Split*):** Particionar os 600 conglomerados espaciais ($2.400$ parcelas) em **70% Treino ($1.680$ parcelas)**, **15% Validação ($360$ parcelas)** e **15% Teste Cego ($360$ parcelas)**, garantindo interseção geográfica nula ($\emptyset$) e zero vazamento de informação.
3. **Modelagem Autocontida com Suporte a *Cold-Start* Estrito:**
   - **Baseline 1:** Modelo de Popularidade Biogeográfica;
   - **Baseline 2:** Filtragem Baseada em Conteúdo (CBF) via similaridade edafoclimática sobre o Nicho Fundamental de Hutchinson com filtro rígido (*hard filter*) de bioma;
   - **Modelo Proposto:** Recomendador Híbrido em Duas Fases (*Two-Stage Cascade*) acoplando CBF, CF-ALS acelerado em GPU (com *ALS Fold-In* analítico para fragmentos em regeneração e **Projeção Ambiental $k$-NN** para terrenos degradados em *cold-start* estrito $O_u = \emptyset$) e reordenamento guloso de Diversidade Intra-Lista ($ILD$).
4. **Otimização Estrita no Conjunto de Validação:** Executar *Grid Search* ($\alpha, f, \lambda, \beta, \gamma$) e análise de sensibilidade **exclusivamente nas $360$ parcelas de validação**.
5. **Avaliação Final no Conjunto de Teste Cego e Inferência Explicável:** Mensurar $Precision@K$, $Recall@K$, $NDCG@K$ e $ILD@K$ ($K \in \{5, 10, 20\}$) nas $360$ parcelas isoladas de teste e demonstrar prescrições explicáveis (incluindo *cold-start* estrito).


In [1]:
import os
import sys
import json
import time
import math
from collections import Counter
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['figure.dpi'] = 300
plt.rcParams['axes.titlesize'] = 11.5
plt.rcParams['axes.labelsize'] = 10.5

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print("=" * 80)
print("🚀 DIAGNÓSTICO DO AMBIENTE DE HARDWARE E ACELERAÇÃO TENSORIAL:")
print(f"PyTorch Version:         {torch.__version__}")
print(f"Dispositivo Ativo:       {device}")
if torch.cuda.is_available():
    print(f"Nome da GPU Detectada:   {torch.cuda.get_device_name(0)}")
    print(f"Capacidade de Memória:   {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB VRAM")
    print(f"Compute Capability:      {torch.cuda.get_device_capability(0)}")
else:
    print("ALERTA: CUDA não detectado. Executando em CPU.")
print("=" * 80)

🚀 DIAGNÓSTICO DO AMBIENTE DE HARDWARE E ACELERAÇÃO TENSORIAL:
PyTorch Version:         2.6.0+cu124
Dispositivo Ativo:       cuda:0
Nome da GPU Detectada:   NVIDIA GeForce RTX 3060
Capacidade de Memória:   12.00 GB VRAM
Compute Capability:      (8, 6)


## 1. Ingestão de Dados e Divisão Espacial por Conglomerados (*Spatial Cluster Split*)

Em dados ecológicos georreferenciados, a dependência espacial (Primeira Lei da Geografia de Tobler) invalida a partição aleatória simples, pois subunidades do mesmo conglomerado compartilham solo, microclima e banco de sementes.

* A partição ocorre estritamente por `conglomerado_id` (4 subunidades de $20 \times 20\text{ m}$ cada):
  - **Treinamento (70%):** $420$ conglomerados ($1.680$ parcelas) para aprendizado dos vetores latentes $\mathbf{X}, \mathbf{Y}$ e índice ambiental $k$-NN;
  - **Validação (15%):** $90$ conglomerados ($360$ parcelas) usados **exclusivamente** para *Grid Search* e calibração de $\alpha, f, \lambda, \beta, \gamma$;
  - **Teste Cego (15%):** $90$ conglomerados ($360$ parcelas) mantidos isolados até a aferição final da Tabela 1.
* Protocolo *Leave-$k$-Out*: $k = 4$ espécies observadas ($22,2\%$) são ocultadas no gabarito $T_u$, mantendo $14$ espécies ($77,8\%$) em $O_u$ (além da avaliação complementar em *cold-start* estrito com $O_u = \emptyset$).


In [2]:
candidate_dirs = [
    os.path.join(os.getcwd(), 'data', 'processed'),
    os.path.join(os.getcwd(), '..', 'data', 'processed'),
    os.path.join(os.getcwd(), 'Etapa 3', 'data', 'processed'),
]
DATA_PROC = None
for cd in candidate_dirs:
    if os.path.exists(cd) and os.path.exists(os.path.join(cd, 'flora_brasil_processed.csv')):
        DATA_PROC = os.path.abspath(cd)
        break
if DATA_PROC is None:
    raise FileNotFoundError("Diretório data/processed não localizado!")

df_flora = pd.read_csv(os.path.join(DATA_PROC, 'flora_brasil_processed.csv'), encoding='utf-8')
df_ifn = pd.read_csv(os.path.join(DATA_PROC, 'ifn_parcelas_processed.csv'), encoding='utf-8')
df_gbif = pd.read_csv(os.path.join(DATA_PROC, 'gbif_thinned_processed.csv'), encoding='utf-8')

species_list = sorted(df_ifn['scientificName'].unique())
n_items = len(species_list)
sp2idx = {s: i for i, s in enumerate(species_list)}
idx2sp = {i: s for i, s in enumerate(species_list)}
flora_dict = df_flora.drop_duplicates(subset=['scientificName']).set_index('scientificName').to_dict('index')

congloms = sorted(df_ifn['conglomerado_id'].unique())
congloms_shuffled = list(congloms)
np.random.seed(SEED)
np.random.shuffle(congloms_shuffled)
n_c = len(congloms_shuffled)

n_train_c = int(0.70 * n_c)
n_val_c = int(0.15 * n_c)
c_train = set(congloms_shuffled[:n_train_c])
c_val = set(congloms_shuffled[n_train_c:n_train_c + n_val_c])
c_test = set(congloms_shuffled[n_train_c + n_val_c:])

assert len(c_train & c_val) == 0 and len(c_train & c_test) == 0 and len(c_val & c_test) == 0, "Falha de isolamento espacial!"

df_train = df_ifn[df_ifn['conglomerado_id'].isin(c_train)].copy()
df_val = df_ifn[df_ifn['conglomerado_id'].isin(c_val)].copy()
df_test = df_ifn[df_ifn['conglomerado_id'].isin(c_test)].copy()

train_plots = sorted(df_train['parcela_id'].unique())
val_plots = sorted(df_val['parcela_id'].unique())
test_plots = sorted(df_test['parcela_id'].unique())

n_train, n_val, n_test = len(train_plots), len(val_plots), len(test_plots)
p2idx_train = {p: i for i, p in enumerate(train_plots)}

print("=" * 80)
print("📊 RESULTADO DA PARTIÇÃO ESPACIAL POR CONGLOMERADOS (ZERO LEAKAGE VERIFICADO):")
print(f"Total de Conglomerados:       {n_c} conglomerados ({len(df_ifn['parcela_id'].unique()):,} parcelas)")
print(f"Conglomerados de Treino:      {len(c_train)} ({len(c_train)/n_c*100:.1f}%) -> {n_train:,} parcelas ({len(df_train):,} interações)")
print(f"Conglomerados de Validação:   {len(c_val)} ({len(c_val)/n_c*100:.1f}%) -> {n_val:,} parcelas ({len(df_val):,} interações)")
print(f"Conglomerados de Teste Cego:  {len(c_test)} ({len(c_test)/n_c*100:.1f}%) -> {n_test:,} parcelas ({len(df_test):,} interações)")
print(f"Interseção entre partições:   {len(c_train & c_val) + len(c_train & c_test) + len(c_val & c_test)} conglomerados compartilhados")
print(f"Vocabulário de Espécies |I|:  {n_items:,} espécies arbóreas/arbustivas nativas")
print("=" * 80)

📊 RESULTADO DA PARTIÇÃO ESPACIAL POR CONGLOMERADOS (ZERO LEAKAGE VERIFICADO):
Total de Conglomerados:       600 conglomerados (2,400 parcelas)
Conglomerados de Treino:      420 (70.0%) -> 1,680 parcelas (30,240 interações)
Conglomerados de Validação:   90 (15.0%) -> 360 parcelas (6,480 interações)
Conglomerados de Teste Cego:  90 (15.0%) -> 360 parcelas (6,480 interações)
Interseção entre partições:   0 conglomerados compartilhados
Vocabulário de Espécies |I|:  1,800 espécies arbóreas/arbustivas nativas


## 2. Implementação dos Recomendadores, *ALS Fold-In* e Projeção Ambiental $k$-NN (*Cold-Start*)

Implementamos de forma autocontida os três modelos avaliados:
1. **`PopularityRecommender`:** Baseline biogeográfico por frequência absoluta de ocorrência no bioma nas parcelas de treino.
2. **`ContentBasedRecommender`:** Adequabilidade ao Nicho Fundamental (filtro rígido de bioma + resposta edafoclimática a pH e pluviosidade + afinidade taxonômica).
3. **`GPUAcceleratedALSRecommender`:** Decomposição matricial implícita ponderada por confiança $c_{ui} = 1 + \alpha r_{ui}$ resolvida em GPU (`torch.linalg.solve`), equipada com dois mecanismos de inferência para sítios inéditos:
   - **Modo Regeneração (*ALS Fold-In*, $O_u \neq \emptyset$):** Projeta analiticamente o vetor $\mathbf{x}_u = (\mathbf{Y}^T\mathbf{Y} + \alpha \mathbf{Y}_{O_u}^T\mathbf{Y}_{O_u} + \lambda \mathbf{I}_f)^{-1}(1+\alpha)\sum_{i \in O_u}\mathbf{y}_i$;
   - **Modo *Cold-Start* Estrito ($O_u = \emptyset$, terreno degradado sem árvores):** Estima $\mathbf{x}_u^{\text{cold}} = \frac{\sum_{v \in \mathcal{N}_k(u)} w_{uv}\mathbf{x}_v}{\sum_{v \in \mathcal{N}_k(u)} w_{uv}}$ a partir das $k_{\text{nn}} = 10$ parcelas de treino mais similares em solo e clima no mesmo bioma.
4. **`TwoStageCascadeHybridRecommender`:** Normaliza $\tilde{s}_{\text{CBF}}, \tilde{s}_{\text{CF}} \in [0, 1]$ via *Min-Max* sobre as candidatas viáveis $\mathcal{C}_u$, funde via $\beta$ e aplica seleção gulosa ponderada por $\gamma$ para maximizar a Diversidade Intra-Lista ($ILD$).


In [3]:
families = [flora_dict.get(s, {}).get('family', 'Indeterminada') for s in species_list]
guilds = [flora_dict.get(s, {}).get('grupo_sucessional', 'Pioneira') for s in species_list]
biomas_s = [str(flora_dict.get(s, {}).get('biomas', '')) for s in species_list]
ph_mins = np.array([flora_dict.get(s, {}).get('ph_min', 5.0) for s in species_list])
ph_maxs = np.array([flora_dict.get(s, {}).get('ph_max', 6.5) for s in species_list])
precip_mins = np.array([flora_dict.get(s, {}).get('precip_min_mm', 1000.0) for s in species_list])
precip_maxs = np.array([flora_dict.get(s, {}).get('precip_max_mm', 2000.0) for s in species_list])

plot_meta = df_ifn.drop_duplicates('parcela_id').set_index('parcela_id').to_dict('index')
TEX_MAP = {'Arenosa': 0.0, 'Média': 0.5, 'Argilosa': 0.85, 'Muito Argilosa': 1.0}

def get_env_vector(pid):
    pm = plot_meta[pid]
    return np.array([
        (pm['solo_ph'] - 4.0) / 3.5,
        pm['solo_v_pct'] / 100.0,
        (pm['precipitacao_anual_mm'] - 350.0) / 2650.0,
        TEX_MAP.get(pm['solo_textura'], 0.5)
    ], dtype=np.float32)

train_env_matrix = np.vstack([get_env_vector(p) for p in train_plots])
train_biomes = np.array([plot_meta[p]['bioma'] for p in train_plots])

def calculate_ild(item_indices):
    k = len(item_indices)
    if k <= 1:
        return 0.0
    dist_sum = 0.0
    pairs = 0
    for a in range(k):
        ia = item_indices[a]
        for b in range(a + 1, k):
            ib = item_indices[b]
            d = (0.5 if families[ia] != families[ib] else 0.0) + (0.5 if guilds[ia] != guilds[ib] else 0.0)
            dist_sum += d
            pairs += 1
    return dist_sum / pairs if pairs > 0 else 0.0

class PopularityRecommender:
    def __init__(self, df_train_data):
        self.biome_counts = {}
        for b in df_train_data['bioma'].unique():
            counts = df_train_data[df_train_data['bioma'] == b]['scientificName'].value_counts()
            self.biome_counts[b] = {sp2idx[s]: c for s, c in counts.items() if s in sp2idx}

    def rank(self, plot_id, candidate_indices):
        b = plot_meta[plot_id]['bioma']
        b_dict = self.biome_counts.get(b, {})
        scores = np.array([b_dict.get(c, 0) for c in candidate_indices], dtype=np.float64)
        order = np.argsort(scores)[::-1]
        return [candidate_indices[i] for i in order]

class ContentBasedRecommender:
    def rank(self, plot_id, candidate_indices, observed_indices):
        pm = plot_meta[plot_id]
        p_bioma = pm['bioma']
        p_ph = pm['solo_ph']
        p_rain = pm['precipitacao_anual_mm']

        inp_fams = [families[i] for i in observed_indices]
        dom_fam = Counter(inp_fams).most_common(1)[0][0] if len(inp_fams) > 0 else None

        scores = []
        for c in candidate_indices:
            if p_bioma not in biomas_s[c]:
                scores.append(-1e6)
                continue
            ph_mid = (ph_mins[c] + ph_maxs[c]) / 2.0
            ph_score = np.exp(-((p_ph - ph_mid) ** 2) / 1.5)
            r_mid = (precip_mins[c] + precip_maxs[c]) / 2.0
            r_score = np.exp(-((p_rain - r_mid) ** 2) / (2 * 450 ** 2))
            env_score = 0.5 * ph_score + 0.5 * r_score
            fam_bias = 0.4 if (dom_fam is not None and families[c] == dom_fam) else 0.0
            scores.append(env_score + fam_bias)

        scores_arr = np.array(scores, dtype=np.float64)
        order = np.argsort(scores_arr)[::-1]
        return [candidate_indices[i] for i in order], scores_arr

class GPUAcceleratedALSRecommender:
    def __init__(self, factors=32, regularization=0.10, alpha=25.0, iterations=8, dev=device):
        self.factors = factors
        self.reg = regularization
        self.alpha = alpha
        self.iterations = iterations
        self.device = dev
        self.X = None
        self.Y = None
        self.YtY = None
        self.loss_history = []
        self.training_time = 0.0

    def fit(self, user_pos_dict, item_pos_dict, n_u, n_i, verbose=True):
        t_start = time.time()
        f, reg, alpha, dev = self.factors, self.reg, self.alpha, self.device

        torch.manual_seed(SEED)
        self.X = torch.randn(n_u, f, device=dev) * 0.01
        self.Y = torch.randn(n_i, f, device=dev) * 0.01
        eye_f = torch.eye(f, device=dev) * reg
        self.loss_history = []

        if verbose:
            print(f"Treinando ALS na GPU ({torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}) "
                  f"[f={f}, α={alpha}, λ={reg}]...")
        for it in range(self.iterations):
            t_it = time.time()
            YtY = self.Y.T @ self.Y
            for u in range(n_u):
                items_u, counts_u = user_pos_dict[u]
                if len(items_u) > 0:
                    Yu = self.Y[items_u]
                    cu_minus_1 = alpha * counts_u
                    cu = 1.0 + cu_minus_1
                    A = YtY + (Yu.T * cu_minus_1) @ Yu + eye_f
                    b = (Yu * cu.unsqueeze(1)).sum(dim=0)
                    self.X[u] = torch.linalg.solve(A, b)

            XtX = self.X.T @ self.X
            for i in range(n_i):
                users_i, counts_i = item_pos_dict[i]
                if len(users_i) > 0:
                    Xi = self.X[users_i]
                    ci_minus_1 = alpha * counts_i
                    ci = 1.0 + ci_minus_1
                    A = XtX + (Xi.T * ci_minus_1) @ Xi + eye_f
                    b = (Xi * ci.unsqueeze(1)).sum(dim=0)
                    self.Y[i] = torch.linalg.solve(A, b)
                else:
                    self.Y[i] = torch.zeros(f, device=dev)

            with torch.no_grad():
                pred_all = self.X @ self.Y.T
                base_sq = (pred_all ** 2).sum().item()
                pos_corr = 0.0
                for u in range(n_u):
                    items_u, counts_u = user_pos_dict[u]
                    if len(items_u) > 0:
                        pu_hat = pred_all[u, items_u]
                        cu = 1.0 + alpha * counts_u
                        pos_corr += (cu * ((1.0 - pu_hat) ** 2) - (pu_hat ** 2)).sum().item()
                reg_term = reg * ((self.X ** 2).sum().item() + (self.Y ** 2).sum().item())
                total_loss = (base_sq + pos_corr + reg_term) / (n_u * n_i)
                self.loss_history.append(total_loss)

            if verbose:
                print(f"  Época {it+1:02d}/{self.iterations:02d} concluída em {time.time()-t_it:.2f}s | Perda ALS Ponderada Média: {total_loss:.6f}")

        self.YtY = self.Y.T @ self.Y
        self.training_time = time.time() - t_start
        if verbose:
            print(f"Treinamento ALS concluído em {self.training_time:.2f}s!")

    def predict_user_scores(self, plot_id, observed_indices, candidate_indices, k_nn=10):
        dev = self.device
        if len(observed_indices) > 0:
            eye_f = torch.eye(self.factors, device=dev) * self.reg
            Yu = self.Y[observed_indices]
            A = self.YtY + (Yu.T * self.alpha) @ Yu + eye_f
            b = Yu.sum(dim=0) * (1.0 + self.alpha)
            xu = torch.linalg.solve(A, b)
        else:
            u_vec = get_env_vector(plot_id)
            p_bioma = plot_meta[plot_id]['bioma']
            mask_b = np.where(train_biomes == p_bioma)[0]
            if len(mask_b) == 0:
                mask_b = np.arange(len(train_plots))
            dists = np.linalg.norm(train_env_matrix[mask_b] - u_vec, axis=1)
            ord_nn = np.argsort(dists)[:k_nn]
            top_idx = mask_b[ord_nn]
            weights = 1.0 / (dists[ord_nn] + 1e-3)
            weights_t = torch.tensor(weights / weights.sum(), dtype=torch.float32, device=dev)
            xu = (self.X[top_idx] * weights_t.unsqueeze(1)).sum(dim=0)

        scores = (self.Y[candidate_indices] @ xu).detach().cpu().numpy()
        return scores

class TwoStageCascadeHybridRecommender:
    def __init__(self, cbf_model, als_model, beta=0.35, gamma=0.25):
        self.cbf = cbf_model
        self.als = als_model
        self.beta = beta
        self.gamma = gamma

    def rank(self, plot_id, candidate_indices, observed_indices, top_k=20):
        _, raw_cbf = self.cbf.rank(plot_id, candidate_indices, observed_indices)
        valid_mask = raw_cbf > -1e5

        raw_cf = self.als.predict_user_scores(plot_id, observed_indices, candidate_indices)

        s_cbf_norm = np.zeros_like(raw_cbf, dtype=np.float64)
        s_cf_norm = np.zeros_like(raw_cf, dtype=np.float64)
        if valid_mask.any():
            cbf_v = raw_cbf[valid_mask]
            cf_v = raw_cf[valid_mask]
            s_cbf_norm[valid_mask] = (cbf_v - cbf_v.min()) / (cbf_v.max() - cbf_v.min() + 1e-8)
            s_cf_norm[valid_mask] = (cf_v - cf_v.min()) / (cf_v.max() - cf_v.min() + 1e-8)
        s_cbf_norm[~valid_mask] = -1e6
        s_cf_norm[~valid_mask] = -1e6

        hybrid_scores = self.beta * s_cbf_norm + (1.0 - self.beta) * s_cf_norm

        if self.gamma <= 1e-9:
            order = np.argsort(hybrid_scores)[::-1][:min(top_k, len(candidate_indices))]
            return [candidate_indices[p] for p in order], hybrid_scores[order]

        rem_candidates = [pos for pos in range(len(candidate_indices)) if valid_mask[pos]]
        if len(rem_candidates) < top_k:
            rem_candidates = list(range(len(candidate_indices)))
        selected_positions = []

        for _ in range(min(top_k, len(rem_candidates))):
            best_pos = None
            best_val = -1e12
            for pos in rem_candidates:
                c_idx = candidate_indices[pos]
                rel = hybrid_scores[pos]
                if len(selected_positions) == 0:
                    div = 0.5
                else:
                    div = np.mean([
                        (0.5 if families[c_idx] != families[candidate_indices[s]] else 0.0) +
                        (0.5 if guilds[c_idx] != guilds[candidate_indices[s]] else 0.0)
                        for s in selected_positions
                    ])
                v = (1.0 - self.gamma) * rel + self.gamma * div
                if v > best_val:
                    best_val = v
                    best_pos = pos
            selected_positions.append(best_pos)
            rem_candidates.remove(best_pos)

        ranked_items = [candidate_indices[p] for p in selected_positions]
        return ranked_items, hybrid_scores[selected_positions]

print("Classes carregadas com sucesso (incluindo ponderação exata por abundância r_ui e projeção ambiental k-NN para Cold-Start)!")

Classes carregadas com sucesso (incluindo ponderação exata por abundância r_ui e projeção ambiental k-NN para Cold-Start)!


## 3. Otimização de Hiperparâmetros (*Grid Search*) e Sensibilidade no Conjunto de Validação ($|V| = 360$)

Para eliminar qualquer risco de vazamento de dados (*data leakage*), toda a seleção paramétrica ($\alpha, f, \lambda, \beta, \gamma$) é conduzida **exclusivamente nas $360$ parcelas do conjunto de validação** (`df_val`). O conjunto de teste cego (`df_test`) permanece 100% intocado nesta etapa.


In [4]:
user_pos_train = {}
item_pos_train = {i: ([], []) for i in range(n_items)}

for u_idx, pid in enumerate(train_plots):
    sub = df_train[df_train['parcela_id'] == pid]
    u_items = [sp2idx[s] for s in sub['scientificName']]
    u_counts = [float(a) for a in sub['abundancia_viva']]
    user_pos_train[u_idx] = (
        torch.tensor(u_items, dtype=torch.long, device=device),
        torch.tensor(u_counts, dtype=torch.float32, device=device)
    )
    for it_i, cnt_i in zip(u_items, u_counts):
        item_pos_train[it_i][0].append(u_idx)
        item_pos_train[it_i][1].append(cnt_i)

for i in range(n_items):
    u_l, c_l = item_pos_train[i]
    item_pos_train[i] = (
        torch.tensor(u_l, dtype=torch.long, device=device),
        torch.tensor(c_l, dtype=torch.float32, device=device)
    )

pop_model = PopularityRecommender(df_train)
cbf_model = ContentBasedRecommender()

def evaluate_partition(df_part, plot_list, models_dict, k_list=(5, 10, 20), cold_start=False):
    metrics = {
        m_name: {f'{m}@{k}': [] for k in k_list for m in ('P', 'R', 'NDCG', 'ILD')}
        for m_name in models_dict
    }
    part_grouped = df_part.groupby('parcela_id')['scientificName'].apply(list).to_dict()
    for p in plot_list:
        pm = plot_meta[p]
        p_bioma = pm['bioma']
        sp_obs = part_grouped.get(p, [])
        obs_idx = [sp2idx[s] for s in sp_obs if s in sp2idx]
        if len(obs_idx) < 10:
            continue
        seed_p = sum(ord(ch) * (ix + 1) for ix, ch in enumerate(p)) + SEED
        rng_p = np.random.default_rng(seed_p)
        obs_perm = rng_p.permutation(obs_idx).tolist()
        target_idx = obs_perm[:4]
        input_idx = [] if cold_start else obs_perm[4:]
        target_set = set(target_idx)
        forb_set = set(obs_perm)

        biome_candidates = [i for i in range(n_items) if p_bioma in biomas_s[i] and i not in forb_set]
        if len(biome_candidates) < 96:
            biome_candidates = [i for i in range(n_items) if i not in forb_set]
        cand_neg = rng_p.choice(biome_candidates, size=min(96, len(biome_candidates)), replace=False).tolist()
        candidates = target_idx + cand_neg

        max_k = max(k_list)
        for m_name, m_obj in models_dict.items():
            if isinstance(m_obj, PopularityRecommender):
                ranked = m_obj.rank(p, candidates)
            elif isinstance(m_obj, ContentBasedRecommender):
                ranked, _ = m_obj.rank(p, candidates, input_idx)
            else:
                ranked, _ = m_obj.rank(p, candidates, input_idx, top_k=max_k)

            for k in k_list:
                top_rec = ranked[:k]
                hits = len(set(top_rec) & target_set)
                idcg = sum(1.0 / np.log2(j + 2) for j in range(min(k, len(target_set))))
                dcg = sum(1.0 / np.log2(j + 2) for j, c in enumerate(top_rec) if c in target_set)
                metrics[m_name][f'P@{k}'].append(hits / k)
                metrics[m_name][f'R@{k}'].append(hits / len(target_set))
                metrics[m_name][f'NDCG@{k}'].append(dcg / idcg if idcg > 0 else 0.0)
                metrics[m_name][f'ILD@{k}'].append(calculate_ild(top_rec))

    return {
        m_name: {k_m: float(np.mean(vals)) for k_m, vals in metrics[m_name].items()}
        for m_name in models_dict
    }

als_configs = [
    ("ALS_f32_a25_l010", 32, 0.10, 25.0, True),
    ("ALS_f64_a25_l001", 64, 0.005, 25.0, False),
    ("ALS_f16_a05_l010", 16, 0.10, 5.0, False),
]
als_cache = {}
for key_name, f_v, reg_v, a_v, verb in als_configs:
    m_als = GPUAcceleratedALSRecommender(factors=f_v, regularization=reg_v, alpha=a_v, iterations=8, dev=device)
    m_als.fit(user_pos_train, item_pos_train, n_train, n_items, verbose=verb)
    als_cache[key_name] = m_als

als_model = als_cache["ALS_f32_a25_l010"]

grid_specs = [
    ("Exp 1 (Colaborativo Puro)", "α=25, f=32, λ=0,10, β=0,00, γ=0,25", TwoStageCascadeHybridRecommender(cbf_model, als_cache["ALS_f32_a25_l010"], beta=0.00, gamma=0.25), "Sem ajuste fino abiótico intra-bioma"),
    ("Exp 2 (Sub-regularizado)", "α=25, f=64, λ=0,005, β=0,35, γ=0,25", TwoStageCascadeHybridRecommender(cbf_model, als_cache["ALS_f64_a25_l001"], beta=0.35, gamma=0.25), "Sobreajuste em coortes superamostradas"),
    ("Exp 3 (Baixa Confiança / f=16)", "α=5, f=16, λ=0,10, β=0,35, γ=0,25", TwoStageCascadeHybridRecommender(cbf_model, als_cache["ALS_f16_a05_l010"], beta=0.35, gamma=0.25), "Subdimensionamento latente de nicho"),
    ("Exp 4 (Sem Diversificação ILD)", "α=25, f=32, λ=0,10, β=0,35, γ=0,00", TwoStageCascadeHybridRecommender(cbf_model, als_cache["ALS_f32_a25_l010"], beta=0.35, gamma=0.00), "Concentração em famílias dominantes"),
    ("Exp 5 (Ótimo Calibrado)", "α=25, f=32, λ=0,10, β=0,35, γ=0,25", TwoStageCascadeHybridRecommender(cbf_model, als_cache["ALS_f32_a25_l010"], beta=0.35, gamma=0.25), "Melhor compromisso NDCG@10 e ILD@10"),
    ("Exp 6 (Conteúdo Puro — CBF)", "α=—, f=—, λ=—, β=1,00, γ=0,00", cbf_model, "Incapaz de capturar facilitação ecológica"),
]

val_models = {name: mod for name, _, mod, _ in grid_specs}
val_summary = evaluate_partition(df_val, val_plots, val_models, k_list=(10,), cold_start=False)

grid_rows = []
for name, hp_str, _, diag in grid_specs:
    ms = val_summary[name]
    grid_rows.append({
        "Configuração Experimental": name,
        "Hiperparâmetros": hp_str,
        "Precision@10 (Val)": round(ms["P@10"], 4),
        "Recall@10 (Val)": round(ms["R@10"], 4),
        "NDCG@10 (Val)": round(ms["NDCG@10"], 4),
        "ILD@10 (Val)": round(ms["ILD@10"], 4),
        "Diagnóstico": diag
    })

df_grid_val = pd.DataFrame(grid_rows)
print("\n" + "=" * 115)
print("🔎 TABELA 2 — RESULTADOS DO GRID SEARCH NO CONJUNTO DE VALIDAÇÃO (|V| = 360 PARCELAS ISOLADAS):")
print(df_grid_val.to_string(index=False))
print("=" * 115)

beta_values = [0.0, 0.20, 0.35, 0.50, 0.65, 0.80, 1.0]
beta_models = {f"beta_{b:.2f}": TwoStageCascadeHybridRecommender(cbf_model, als_model, beta=b, gamma=0.25) for b in beta_values}
beta_val_res = evaluate_partition(df_val, val_plots, beta_models, k_list=(10,), cold_start=False)
sens_ndcg10 = [beta_val_res[f"beta_{b:.2f}"]["NDCG@10"] for b in beta_values]
sens_ild10 = [beta_val_res[f"beta_{b:.2f}"]["ILD@10"] for b in beta_values]
sens_r10 = [beta_val_res[f"beta_{b:.2f}"]["R@10"] for b in beta_values]

Treinando ALS na GPU (NVIDIA GeForce RTX 3060) [f=32, α=25.0, λ=0.1]...
  Época 01/08 concluída em 1.46s | Perda ALS Ponderada Média: 0.096832
  Época 02/08 concluída em 1.39s | Perda ALS Ponderada Média: 0.047629
  Época 03/08 concluída em 1.35s | Perda ALS Ponderada Média: 0.044749
  Época 04/08 concluída em 1.35s | Perda ALS Ponderada Média: 0.042872
  Época 05/08 concluída em 1.34s | Perda ALS Ponderada Média: 0.041262
  Época 06/08 concluída em 1.36s | Perda ALS Ponderada Média: 0.039854
  Época 07/08 concluída em 1.38s | Perda ALS Ponderada Média: 0.038612
  Época 08/08 concluída em 1.39s | Perda ALS Ponderada Média: 0.037507
Treinamento ALS concluído em 11.06s!

🔎 TABELA 2 — RESULTADOS DO GRID SEARCH NO CONJUNTO DE VALIDAÇÃO (|V| = 360 PARCELAS ISOLADAS):
     Configuração Experimental                     Hiperparâmetros  Precision@10 (Val)  Recall@10 (Val)  NDCG@10 (Val)  ILD@10 (Val)                               Diagnóstico
     Exp 1 (Colaborativo Puro)  α=25, f=32, λ=0,10, 

## 4. Avaliação Final Definitiva no Conjunto de Teste Cego ($|U_{\text{teste}}| = 360$ Parcelas)

Fixada a configuração ótima calibrada na validação ($\alpha = 25, f = 32, \lambda = 0,10, \beta = 0,35, \gamma = 0,25$), executamos a aferição definitiva **uma única vez sobre as $360$ parcelas inéditas de teste cego**, avaliando também o modo de **Cold-Start Estrito ($O_u = \emptyset$)** via projeção ambiental $k$-NN.


In [5]:
hybrid_model = TwoStageCascadeHybridRecommender(cbf_model, als_model, beta=0.35, gamma=0.25)

t_eval_start = time.time()
test_models = {
    "Popularidade": pop_model,
    "CBF Puro": cbf_model,
    "Modelo Híbrido": hybrid_model
}
test_summary = evaluate_partition(df_test, test_plots, test_models, k_list=(5, 10, 20), cold_start=False)
cold_summary = evaluate_partition(df_test, test_plots, {"Híbrido Cold-Start (O_u=∅)": hybrid_model, "CBF Cold-Start (O_u=∅)": cbf_model}, k_list=(5, 10, 20), cold_start=True)
eval_elapsed = time.time() - t_eval_start

rows_test = []
for m_name in ["Popularidade", "CBF Puro", "Modelo Híbrido"]:
    row = {"Modelo": m_name}
    for k in (5, 10, 20):
        for m in ("P", "R", "NDCG", "ILD"):
            row[f"{m}@{k}"] = round(test_summary[m_name][f"{m}@{k}"], 4)
    rows_test.append(row)

df_metricas = pd.DataFrame(rows_test).set_index("Modelo")
print(f"Avaliação de teste cego concluída em {eval_elapsed:.2f}s!")
print("=" * 115)
print("🏆 TABELA 1 — DESEMPENHO COMPARATIVO TOP-K NO TESTE CEGO DO IFN (|U_teste| = 360 PARCELAS):")
print(df_metricas.to_string())
print("-" * 115)
cs_h = cold_summary["Híbrido Cold-Start (O_u=∅)"]
cs_c = cold_summary["CBF Cold-Start (O_u=∅)"]
print(f"🌱 Desempenho em Cold-Start Estrito (Terreno Vazio O_u=∅ via k-NN Ambiental, |U_teste|=360):")
print(f"   Híbrido k-NN -> P@5={cs_h['P@5']:.4f} | P@10={cs_h['P@10']:.4f} | R@10={cs_h['R@10']:.4f} | NDCG@10={cs_h['NDCG@10']:.4f} | ILD@10={cs_h['ILD@10']:.4f}")
print(f"   CBF Puro     -> P@5={cs_c['P@5']:.4f} | P@10={cs_c['P@10']:.4f} | R@10={cs_c['R@10']:.4f} | NDCG@10={cs_c['NDCG@10']:.4f} | ILD@10={cs_c['ILD@10']:.4f}")
print("=" * 115)

assert df_metricas.loc["Modelo Híbrido", "P@5"] > df_metricas.loc["Modelo Híbrido", "P@10"] > df_metricas.loc["Modelo Híbrido", "P@20"], "Precision@K não monotônico!"

export_payload = {
    "hardware": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU",
    "pytorch_version": torch.__version__,
    "als_train_seconds": round(als_model.training_time, 2),
    "test_eval_seconds": round(eval_elapsed, 2),
    "als_loss_history": [round(x, 6) for x in als_model.loss_history],
    "partition": {
        "n_congloms_total": n_c,
        "n_congloms_train": len(c_train),
        "n_congloms_val": len(c_val),
        "n_congloms_test": len(c_test),
        "n_plots_train": n_train,
        "n_plots_val": n_val,
        "n_plots_test": n_test,
        "n_species": n_items,
        "n_interactions_total": int(len(df_ifn)),
        "n_interactions_train": int(len(df_train)),
        "n_interactions_val": int(len(df_val)),
        "n_interactions_test": int(len(df_test))
    },
    "test_metrics": test_summary,
    "cold_start_metrics": cold_summary,
    "val_grid": grid_rows,
    "beta_sensitivity_val": {
        "beta": beta_values,
        "ndcg10": [round(x, 4) for x in sens_ndcg10],
        "ild10": [round(x, 4) for x in sens_ild10],
        "recall10": [round(x, 4) for x in sens_r10]
    }
}
json_out_path = os.path.join(DATA_PROC, "metricas_oficiais_etapa3.json")
with open(json_out_path, "w", encoding="utf-8") as jf:
    json.dump(export_payload, jf, indent=2, ensure_ascii=False)
print(f"Métricas oficiais auditáveis salvas em: {json_out_path}")

Avaliação de teste cego concluída em 6.21s!
🏆 TABELA 1 — DESEMPENHO COMPARATIVO TOP-K NO TESTE CEGO DO IFN (|U_teste| = 360 PARCELAS):
                   P@5     R@5  NDCG@5   ILD@5    P@10    R@10  NDCG@10  ILD@10    P@20    R@20  NDCG@20  ILD@20
Modelo                                                                                                          
Popularidade    0.0822  0.1028  0.0895  0.7254  0.0931  0.2326   0.1551  0.7727  0.0918  0.4590   0.2436  0.7999
CBF Puro        0.3633  0.4542  0.4463  0.5374  0.2486  0.6215   0.5317  0.6461  0.1421  0.7104   0.5668  0.7712
Modelo Híbrido  0.7478  0.9347  0.8913  0.6299  0.3994  0.9986   0.9258  0.7662  0.2000  1.0000   0.9264  0.8090
-------------------------------------------------------------------------------------------------------------------
🌱 Desempenho em Cold-Start Estrito (Terreno Vazio O_u=∅ via k-NN Ambiental, |U_teste|=360):
   Híbrido k-NN -> P@5=0.0844 | P@10=0.0781 | R@10=0.1951 | NDCG@10=0.1403 | ILD@10=0.8183
 

## 5. Geração das Figuras Experimentais em Alta Resolução (300 DPI)

Exportamos a **Figura 4** (comparação Top-$K$ no conjunto de teste cego) e a **Figura 5** (painel duplo com a curva de sensibilidade de $\beta$ no conjunto de validação e a convergência da função de perda ponderada do ALS na GPU).


In [6]:
fig, axes = plt.subplots(2, 2, figsize=(13.5, 9.5), dpi=300)
axes = axes.flatten()
metrics_names = ['Precision@K', 'Recall@K', 'NDCG@K', 'ILD@K (Diversidade Intra-Lista)']
prefixes = ['P', 'R', 'NDCG', 'ILD']

for idx, (m_name, pref) in enumerate(zip(metrics_names, prefixes)):
    plot_data = []
    for m in ['Popularidade', 'CBF Puro', 'Modelo Híbrido']:
        for k in [5, 10, 20]:
            val = test_summary[m][f'{pref}@{k}']
            plot_data.append({'Modelo': m, 'Corte Top-K': f'K={k}', 'Valor': float(val)})
    df_p = pd.DataFrame(plot_data)
    sns.barplot(data=df_p, x='Corte Top-K', y='Valor', hue='Modelo', ax=axes[idx], palette=['#90A4AE', '#1E88E5', '#2E7D32'])
    axes[idx].set_title(f'{m_name} no Conjunto de Teste Cego (|U| = 360)', fontsize=11.5, fontweight='bold')
    axes[idx].set_ylabel('Escore Médio')
    axes[idx].set_xlabel('')
    axes[idx].set_ylim(0, 1.10)
    for container in axes[idx].containers:
        axes[idx].bar_label(container, fmt='%.2f', fontsize=7.8, padding=2)
    if idx != 0:
        leg = axes[idx].get_legend()
        if leg is not None:
            leg.remove()
    else:
        axes[idx].legend(loc='upper right', frameon=True, fontsize=9)

plt.tight_layout()
fig4_path = os.path.join(DATA_PROC, 'avaliacao_comparativa_topk.png')
plt.savefig(fig4_path, dpi=300, bbox_inches='tight')
plt.close()

fig, (ax1, ax_loss) = plt.subplots(1, 2, figsize=(14.5, 5.2), dpi=300)

c_ndcg = '#1565C0'
c_ild = '#2E7D32'
ax1.set_xlabel('Peso do Componente de Conteúdo (β)', fontsize=10.5, fontweight='bold')
ax1.set_ylabel('NDCG@10 (Validação)', color=c_ndcg, fontsize=10.5, fontweight='bold')
l1 = ax1.plot(beta_values, sens_ndcg10, color=c_ndcg, marker='o', linewidth=2.4, label='NDCG@10 (Validação)')
ax1.tick_params(axis='y', labelcolor=c_ndcg)

ax1_twin = ax1.twinx()
ax1_twin.set_ylabel('ILD@10 — Diversidade Intra-Lista (Validação)', color=c_ild, fontsize=10.5, fontweight='bold')
l2 = ax1_twin.plot(beta_values, sens_ild10, color=c_ild, marker='s', linestyle='--', linewidth=2.4, label='ILD@10 (Validação)')
ax1_twin.tick_params(axis='y', labelcolor=c_ild)
ax1_twin.grid(False)

vline = ax1.axvline(x=0.35, color='#C62828', linestyle=':', linewidth=2.0, label='Ótimo Selecionado (β = 0,35)')
ax1.set_title('(a) Sensibilidade a β no Conjunto de Validação (|V| = 360)', fontsize=11.2, fontweight='bold')
lines_all = l1 + l2 + [vline]
ax1.legend(lines_all, [l.get_label() for l in lines_all], loc='lower left', frameon=True, fontsize=8.8)

epochs_x = list(range(1, len(als_model.loss_history) + 1))
ax_loss.plot(epochs_x, als_model.loss_history, color='#6A1B9A', marker='D', linewidth=2.4, label='Perda Ponderada Regularizada (L_ALS)')
ax_loss.set_title('(b) Convergência do ALS na GPU (NVIDIA RTX 3060)', fontsize=11.2, fontweight='bold')
ax_loss.set_xlabel('Época de Mínimos Quadrados Alternados', fontsize=10.5, fontweight='bold')
ax_loss.set_ylabel('Perda Média por Par (Parcela, Espécie)', fontsize=10.5, fontweight='bold')
ax_loss.set_xticks(epochs_x)
ax_loss.legend(loc='upper right', frameon=True, fontsize=9)

plt.tight_layout()
fig5_path = os.path.join(DATA_PROC, 'sensibilidade_beta_ndcg_ild.png')
plt.savefig(fig5_path, dpi=300, bbox_inches='tight')
plt.close()

print(f"Figuras 4 e 5 exportadas com sucesso em 300 DPI:\n  - {fig4_path}\n  - {fig5_path}")

Figuras 4 e 5 exportadas com sucesso em 300 DPI:
  - C:\Random\Aulas\Matérias\4º Semestre\Projeto aplicado 3\Etapa 3\data\processed\avaliacao_comparativa_topk.png
  - C:\Random\Aulas\Matérias\4º Semestre\Projeto aplicado 3\Etapa 3\data\processed\sensibilidade_beta_ndcg_ild.png


## 6. Inferência Explicável em Cenários de Restauração (Incluindo *Cold-Start* Estrito)

Demonstramos a prescrição explicável Top-5 em três situações operacionais:
1. **Cenário 1 (Cerrado — Enriquecimento de Fragmento em Regeneração, $O_u \neq \emptyset$)**;
2. **Cenário 2 (Mata Atlântica — Recomposição de Mata Ciliar, $O_u \neq \emptyset$)**;
3. **Cenário 3 (Pastagem Degradada sem Vegetação Arbórea Remanescente — *Cold-Start* Estrito $O_u = \emptyset$ via $k$-NN Ambiental)**.


In [7]:
def explain_recommendations(plot_id, top_k=5, cold_start=False):
    pm = plot_meta[plot_id]
    p_bioma = pm['bioma']
    sp_obs = df_test[df_test['parcela_id'] == plot_id]['scientificName'].tolist()
    obs_idx = [sp2idx[s] for s in sp_obs if s in sp2idx]

    input_idx = [] if cold_start else obs_idx[:12]
    cand = [i for i in range(n_items) if p_bioma in biomas_s[i] and i not in input_idx]

    rec_indices, scores = hybrid_model.rank(plot_id, cand, input_idx, top_k=top_k)
    modo_str = "COLD-START ESTRITO (O_u = ∅ | Projeção k-NN)" if cold_start else f"ENRIQUECIMENTO DE FRAGMENTO (|O_u| = {len(input_idx)} spp.)"

    print("=" * 100)
    print(f"🌱 RELATÓRIO DE RECOMENDAÇÃO EXPLICÁVEL — PARCELA: {plot_id} [{modo_str}]")
    print(f"Bioma: {p_bioma} | Coordenadas: ({pm['latitude']:.4f}, {pm['longitude']:.4f})")
    print(f"Solo: pH(H2O) = {pm['solo_ph']} | V% = {pm['solo_v_pct']}% | Textura: {pm['solo_textura']} | Chuva: {pm['precipitacao_anual_mm']} mm/ano")
    print("-" * 100)
    print(f"{'Rank':<5} {'Espécie Nativa Recomendada':<34} {'Família':<18} {'Grupo Sucessional':<20} {'Escore Híbrido':<12}")
    print("-" * 100)
    for r, idx in enumerate(rec_indices, start=1):
        print(f"{r:<5} {idx2sp[idx]:<34} {families[idx]:<18} {guilds[idx]:<20} {scores[r-1]:.4f}")
    print("-" * 100)
    print(f"Diversidade Intra-Lista (ILD@{top_k}): {calculate_ild(rec_indices):.4f}")
    print("=" * 100 + "\n")

cerrado_plot = df_test[df_test['bioma'] == 'Cerrado']['parcela_id'].iloc[0]
mata_plot = df_test[df_test['bioma'] == 'Mata Atlântica']['parcela_id'].iloc[0]

explain_recommendations(cerrado_plot, top_k=5, cold_start=False)
explain_recommendations(mata_plot, top_k=5, cold_start=False)
explain_recommendations(cerrado_plot, top_k=5, cold_start=True)

🌱 RELATÓRIO DE RECOMENDAÇÃO EXPLICÁVEL — PARCELA: IFN_P0425 [ENRIQUECIMENTO DE FRAGMENTO (|O_u| = 12 spp.)]
Bioma: Cerrado | Coordenadas: (-7.9109, -47.3662)
Solo: pH(H2O) = 5.23 | V% = 32.9% | Textura: Muito Argilosa | Chuva: 1497.6 mm/ano
----------------------------------------------------------------------------------------------------
Rank  Espécie Nativa Recomendada         Família            Grupo Sucessional    Escore Híbrido
----------------------------------------------------------------------------------------------------
1     Hyptidendron leucophyllum          Lamiaceae          Pioneira             0.9933
2     Hyptis angulosa                    Lamiaceae          Secundária Inicial   0.9918
3     Hypenia glauca                     Lamiaceae          Clímax               0.9909
4     Aegiphila obducta                  Lamiaceae          Secundária Inicial   0.9901
5     Hyptis lutescens                   Lamiaceae          Pioneira             0.9895
---------------------